In [0]:
import requests
import json
from pyspark.sql import SparkSession

#Definition des variables

In [0]:
CLIENT_ID = ""
CLIENT_SECRET = "I"
USER_ACCESS_TYPE = "TOAST_MACHINE_CLIENT"
main_url = "https://ws-api.toasttab.com"




#recuperation du token

In [0]:
auth_url = f"{main_url}/authentication/v1/authentication/login"

auth_payload = {
    "clientId": CLIENT_ID,
    "clientSecret": CLIENT_SECRET,
    "userAccessType": USER_ACCESS_TYPE
}

auth_headers = {
    "Content-Type": "application/json"
}

auth_response = requests.post(auth_url, headers=auth_headers, json=auth_payload)

print("Status auth :", auth_response.status_code)
print(auth_response.text)

In [0]:
auth_data = auth_response.json()
access_token = auth_data["token"]["accessToken"]

print("Token récupéré avec succès")

#recuperation des id de restaurant

In [0]:
restaurants_url = f"{main_url}/partners/v1/restaurants"

restaurants_headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

restaurants_response = requests.get(
    restaurants_url,
    headers=restaurants_headers
)

print("Status restaurants :", restaurants_response.status_code)
print(restaurants_response.text[:1000])

#transformer la reponse en JSON

In [0]:
restaurants_list = restaurants_response.json()

print("Nombre de restaurants trouvés :", len(restaurants_list))
print("Premier restaurant :")
print(restaurants_list[0])

#affichage des id et les noms des restaurants

In [0]:
from pyspark.sql import functions as F

for restaurant in restaurants_list[:10]:
    print("restaurantGuid :", restaurant["restaurantGuid"])
    print("restaurantName :", restaurant["restaurantName"])
    print("-" * 50)

#la recuperation de chaque details des restaurant

In [0]:
all_restaurant_details = []

for restaurant in restaurants_list:
    restaurant_guid = restaurant["restaurantGuid"]
    restaurant_name = restaurant["restaurantName"]

    detail_url = f"{main_url}/restaurants/v1/restaurants/{restaurant_guid}"

    detail_headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "Toast-Restaurant-External-ID": restaurant_guid
    }

    try:
        detail_response = requests.get(detail_url, headers=detail_headers)
        detail_response.raise_for_status()

        detail_data = detail_response.json()

        all_restaurant_details.append({
            "restaurant_guid": restaurant_guid,
            "restaurant_name": restaurant_name,
            "raw_json": json.dumps(detail_data)
        })

        print(f"OK : {restaurant_guid} - {restaurant_name}")

    except Exception as e:
        print(f"Erreur pour {restaurant_guid} - {restaurant_name} : {str(e)}")

In [0]:
df = spark.createDataFrame(all_restaurant_details) \
    .withColumn("ingestion_timestamp", F.current_timestamp())


In [0]:
df.write.mode("overwrite").saveAsTable("project1.bronze.toast_restaurant_raw")

In [0]:
%sql
select * from project1.bronze.toast_restaurant_raw